# Correction J2 · La file à la friterie

Au début d’un créneau, on connaît le nombre de commandes en file. Peut-on en déduire si le prochain client attendra plus de 15 minutes ? La classe **1** indique une longue attente, la classe **0** une attente de 15 minutes ou moins.

## Comprendre · Du nombre de commandes au score

Imaginons une feuille d’arbre contenant quatre anciens créneaux avec des files de neuf commandes ou plus. Trois clients ont attendu plus de 15 minutes. L’arbre attribuerait le même score, **3/4 = 0,75**, à tout nouveau créneau qui tombe dans cette feuille. La régression logistique, elle, apprend une courbe en S. Elle peut donner des scores différents à des files de neuf et de dix commandes.

## Visualiser · Quatre créneaux à la main

Voici quatre scores fictifs. Pour décider, on retient les créneaux dont le score **atteint ou dépasse** le seuil.

| Créneau | A | B | C | D |
|---|---:|---:|---:|---:|
| Score de longue attente | 0,12 | 0,37 | 0,58 | 0,84 |
| Classe observée | 0 | 1 | 0 | 1 |

À **0,50**, on retient C et D. C est une fausse alerte. À **0,70**, on retient seulement D. B est une longue attente manquée dans les deux cas.

## Approfondir · Lire les erreurs

Au seuil de **0,70**, les lignes donnent ce qui s’est passé et les colonnes ce que le modèle a proposé.

| Classe observée | Proposé 0 | Proposé 1 |
|---|---:|---:|
| 0 | 2 (A, C) | 0 |
| 1 | 1 (B) | 1 (D) |

- **Exactitude** — trois créneaux bien classés sur quatre, soit **75 %**.
- **Précision** — une longue attente parmi une alerte, soit **100 %**.
- **Rappel** — une longue attente retrouvée sur deux, soit **50 %**.
- **F1** — deux fois le nombre de vrais positifs, divisé par ce même nombre augmenté des faux positifs et des faux négatifs : **2/(2 + 0 + 1) ≈ 67 %**.

## Coder · Charger les données et réserver le dernier service

On garde les douze créneaux du dernier service pour la décision de terrain. Les autres services fournissent les créneaux à partager comme dans le cours.

In [1]:
from pathlib import Path
import pandas as pd

dossier_data = next(d for d in (Path("data"), Path("../data"), Path("../../data")) if d.is_dir())
donnees = pd.read_csv(dossier_data / "friterie.csv")
print(len(donnees), "créneaux chargés.")
print(donnees.dtypes)
print(donnees.isna().sum())
print("Créneaux en double :", donnees.duplicated(["date_service", "heure"]).sum())
print("Classes observées :", donnees["attente_longue"].unique())

720 créneaux chargés.
date_service            object
heure                   object
commandes_en_attente     int64
attente_longue           int64
dtype: object
date_service            0
heure                   0
commandes_en_attente    0
attente_longue          0
dtype: int64
Créneaux en double : 0
Classes observées : [0 1]


In [2]:
derniere_date = donnees["date_service"].max()
dernier_service = donnees[donnees["date_service"] == derniere_date].sort_values("heure")
historique = donnees[donnees["date_service"] < derniere_date]

X = historique[["commandes_en_attente"]]
y = historique["attente_longue"]

### Réserver les créneaux pour chaque étape

Le premier partage réserve **20 %** des créneaux pour le test. Le second réserve **25 % des 80 % restants** pour la validation, soit **20 % du total**. Il reste environ **60 %** pour apprendre. `stratify` conserve des proportions voisines de longues attentes dans les trois lots. La graine 42 permet de retrouver le même tirage.

In [3]:
from sklearn.model_selection import train_test_split

def train_test_val_split(X, y):
    X_dev, X_test, y_dev, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=42
    )
    X_ap, X_val, y_ap, y_val = train_test_split(
        X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=42
    )
    return X_test, y_test, X_ap, X_val, y_ap, y_val

X_test, y_test, X_ap, X_val, y_ap, y_val = train_test_val_split(X, y)

print("Apprentissage :", len(X_ap))
print("Validation :", len(X_val), "créneaux, dont", y_val.sum(), "longues attentes.")
print("Test :", len(X_test))

Apprentissage : 424
Validation : 142 créneaux, dont 32 longues attentes.
Test : 142


In [4]:
print(f"Longues attentes à l’apprentissage : {y_ap.mean():.1%}")
tableau = X_ap.copy()
tableau["attente_longue"] = y_ap
print(tableau.groupby("commandes_en_attente")["attente_longue"].mean().round(2))

Longues attentes à l’apprentissage : 22.2%
commandes_en_attente
0     0.00
1     0.00
2     0.00
3     0.03
4     0.00
5     0.08
6     0.05
7     0.13
8     0.29
9     0.44
10    0.68
11    0.64
12    0.65
13    0.86
14    0.86
15    0.83
Name: attente_longue, dtype: float64


**Règle de référence.** Sur les 142 créneaux de validation, 32 sont de classe 1. Répondre toujours 0 donne 110 bonnes réponses sur 142, soit environ 77 % d’exactitude. Le rappel vaut 0 % puisque les 32 longues attentes sont manquées.

### Entraîner l’arbre et la régression logistique

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text

arbre = DecisionTreeClassifier(max_depth=1, random_state=42)
logistique = LogisticRegression(max_iter=1000)

arbre.fit(X_ap, y_ap)
logistique.fit(X_ap, y_ap)

print(export_text(arbre, feature_names=["commandes en attente"], show_weights=True))
print("Ordre des classes :", logistique.classes_)

|--- commandes en attente <= 8.50
|   |--- weights: [288.00, 22.00] class: 0
|--- commandes en attente >  8.50
|   |--- weights: [42.00, 72.00] class: 1

Ordre des classes : [0 1]


L’arbre pose une seule question. Jusqu’à huit commandes, 22 créneaux sur 310 sont de classe 1, soit un score de 0,071. À partir de neuf commandes, il y en a 72 sur 114, soit 0,632. Vérifions ce saut sur deux files voisines.

In [6]:
exemples = pd.DataFrame({"commandes_en_attente": [2, 7, 8, 9, 12]})
comparaison = exemples.copy()
comparaison["Score arbre"] = arbre.predict_proba(exemples)[:, 1]
comparaison["Score logistique"] = logistique.predict_proba(exemples)[:, 1]
comparaison["Classe proposée par l’arbre"] = arbre.predict(exemples)
print(comparaison.round(3).to_string(index=False))

 commandes_en_attente  Score arbre  Score logistique  Classe proposée par l’arbre
                    2        0.071             0.010                            0
                    7        0.071             0.161                            0
                    8        0.071             0.257                            0
                    9        0.632             0.384                            1
                   12        0.632             0.783                            1


### Essayer trois seuils pour chacun des deux modèles

Trois renforts sur douze créneaux représentent un quart d’un service. Pour comparer les règles sur le lot mélangé de validation, on prend donc un budget global de **25 %**. La limite réelle de trois appels sera appliquée dans l’ordre sur le dernier service.

In [7]:
from sklearn.metrics import precision_score, recall_score

seuils = [0.20, 0.50, 0.70]
modeles = {"Logistique": logistique, "Arbre": arbre}
essais = []

for nom, modele in modeles.items():
    scores = modele.predict_proba(X_val)[:, 1]
    for seuil in seuils:
        retenues = scores >= seuil
        essais.append({
            "modèle": nom,
            "seuil": seuil,
            "alertes": int(retenues.sum()),
            "1 retrouvés": int(y_val[retenues].sum()),
            "précision": precision_score(y_val, retenues, zero_division=0),
            "rappel": recall_score(y_val, retenues),
        })

resultats = pd.DataFrame(essais)
print(resultats.round(2).to_string(index=False))

    modèle  seuil  alertes  1 retrouvés  précision  rappel
Logistique    0.2       41           26       0.63    0.81
Logistique    0.5       23           20       0.87    0.62
Logistique    0.7       10            9       0.90    0.28
     Arbre    0.2       31           21       0.68    0.66
     Arbre    0.5       31           21       0.68    0.66
     Arbre    0.7        0            0       0.00    0.00


### Choisir une règle dans la limite des places

On garde les règles qui respectent le budget. Parmi elles, on cherche celle qui retrouve le plus de longues attentes. À résultat égal, on préfère celle qui déclenche le moins d’alertes.

In [8]:
def choisir_regle(tableau, places):
    possibles = tableau[tableau["alertes"] <= places]
    classement = possibles.sort_values(
        ["1 retrouvés", "alertes"], ascending=[False, True]
    )
    return classement.iloc[0]

places = int(0.25 * len(X_val))
choix = choisir_regle(resultats, places)
print("Places sur la validation :", places)
print(choix.to_string())

Places sur la validation : 35
modèle            Arbre
seuil               0.2
alertes              31
1 retrouvés          21
précision      0.677419
rappel          0.65625


Avec 35 places, l’arbre retrouve **21 longues attentes en 31 alertes**. Les seuils 0,20 et 0,50 de l’arbre donnent les mêmes décisions car ils sont tous deux entre les scores de ses deux feuilles. On peut garder le premier.

### Faire le bilan sur la validation

In [9]:
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

def bilan(y_observe, propositions):
    matrice = confusion_matrix(y_observe, propositions, labels=[0, 1])
    print(pd.DataFrame(matrice, index=["Observé 0", "Observé 1"],
                       columns=["Proposé 0", "Proposé 1"]).to_string())
    for nom, mesure in [("Exactitude", accuracy_score), ("Précision", precision_score),
                        ("Rappel", recall_score), ("F1", f1_score)]:
        print(f"{nom} : {mesure(y_observe, propositions):.1%}")

modele_choisi = modeles[choix["modèle"]]
scores = modele_choisi.predict_proba(X_val)[:, 1]
propositions = (scores >= choix["seuil"]).astype(int)
bilan(y_val, propositions)

           Proposé 0  Proposé 1
Observé 0        100         10
Observé 1         11         21
Exactitude : 85.2%
Précision : 67.7%
Rappel : 65.6%
F1 : 66.7%


Une fausse alerte mobiliserait un renfort pour une attente de 15 minutes ou moins. Il y en a **10**. Une longue attente manquée est un créneau de classe 1 pour lequel le modèle propose 0. Il y en a **11**. La précision est donc de 21/31 et le rappel de 21/32.

## Pratiquer · Que changerait un quatrième renfort ?

Sur les quatre créneaux fictifs, abaisser le seuil à 0,30 retient B, C et D. On retrouve les deux longues attentes et on déclenche une fausse alerte. L’exactitude vaut 3/4, la précision 2/3, le rappel 2/2 et le F1 vaut 4/5.

Sur la validation, passons de trois à quatre places sur douze. On reprend les **six essais déjà calculés**.

In [10]:
places_sup = int(4 / 12 * len(X_val))
autre_choix = choisir_regle(resultats, places_sup)
print(autre_choix.to_string())
print("Alertes en plus :", autre_choix["alertes"] - choix["alertes"])
print("Longues attentes retrouvées en plus :", autre_choix["1 retrouvés"] - choix["1 retrouvés"])

modèle         Logistique
seuil                 0.2
alertes                41
1 retrouvés            26
précision        0.634146
rappel             0.8125
Alertes en plus : 10
Longues attentes retrouvées en plus : 5


### Vérifier la règle à trois renforts sur le test

Le quatrième renfort était une simulation. Pour le bilan demandé par l’exercice, on garde le modèle et le seuil retenus avec le budget initial.

In [11]:
scores_test = modele_choisi.predict_proba(X_test)[:, 1]
propositions_test = (scores_test >= choix["seuil"]).astype(int)
bilan(y_test, propositions_test)

           Proposé 0  Proposé 1
Observé 0        102          9
Observé 1          8         23
Exactitude : 88.0%
Précision : 71.9%
Rappel : 74.2%
F1 : 73.0%


### Appliquer la règle au dernier service

Les créneaux arrivent dans l’ordre des heures. On appelle dès que le score atteint le seuil, puis on arrête les appels dès que les trois places sont utilisées. Les étiquettes servent ensuite à vérifier le résultat.

In [12]:
scores_service = modele_choisi.predict_proba(dernier_service[["commandes_en_attente"]])[:, 1]
alertes = dernier_service[scores_service >= choix["seuil"]]
appels = alertes.head(3)

print(appels[["heure", "commandes_en_attente", "attente_longue"]].to_string(index=False))
avec_appel = appels["attente_longue"].sum()
sans_appel = dernier_service["attente_longue"].sum() - avec_appel
print("Longues attentes avec appel :", avec_appel)
print("Longues attentes sans appel :", sans_appel)

heure  commandes_en_attente  attente_longue
19:15                    12               1
19:30                    16               1
19:45                    11               1
Longues attentes avec appel : 3
Longues attentes sans appel : 1


## Retenir · Expliquer la décision

Le responsable aurait appelé à **19 h 15, 19 h 30 et 19 h 45**. Ces trois créneaux correspondent à de longues attentes observées dans le CSV. À **20 h**, il y avait huit commandes en file. L’arbre a classé ce créneau en 0, alors que le client a attendu plus de 15 minutes. Sur ce service, la règle a donc repéré **trois longues attentes sur quatre**.